# LSU Football Data Science Starter Project

Uses the [CollegeFootballData.com](https://collegefootballdata.com) (CFBD) API to pull LSU Tigers stats and do exploratory data analysis.

**Setup:**
1. Get a free API key at [collegefootballdata.com/key](https://collegefootballdata.com/key)
2. `pip install cfbd pandas matplotlib seaborn scikit-learn`
3. Paste your API key in the config cell below
4. Run the cells in order

## Imports & Configuration

In [ ]:
import cfbd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [ ]:
# =============================================================================
# CONFIG - paste your CFBD API key here
# =============================================================================
API_KEY = "YEFaPC33rtmiiK9gLINyEL4rIACHOuATHfWt9LLvkrecGFHb817upyrBsbVy7cRC"

TEAM = "LSU"
YEARS = range(2015, 2026)  # 10+ seasons of data

configuration = cfbd.Configuration(access_token=API_KEY)

## 1. Pull Game-Level Data

Pulls all LSU games for the specified year range from the CFBD API.

In [ ]:
def get_lsu_games(years=YEARS):
    """Pull all LSU games for the specified year range."""
    all_games = []

    with cfbd.ApiClient(configuration) as api_client:
        games_api = cfbd.GamesApi(api_client)

        for year in years:
            try:
                games = games_api.get_games(year=year, team=TEAM)
                for g in games:
                    all_games.append(g.to_dict())
                print(f"  {year}: {len(games)} games")
            except Exception as e:
                print(f"  {year}: error - {e}")

    df = pd.DataFrame(all_games)
    return df

In [ ]:
print("=" * 60)
print("PULLING LSU GAME DATA")
print("=" * 60)
games_df = get_lsu_games()
print(f"\nTotal games pulled: {len(games_df)}")
print(f"Columns: {list(games_df.columns)}")

## 2. Feature Engineering

Derives LSU points, opponent points, margin, win/loss flag, home/away indicator, and opponent name from the raw game records.

In [ ]:
def engineer_features(df):
    """
    Add useful columns: LSU score, opponent score, margin, home/away flag,
    win/loss indicator, etc.
    """
    df = df.copy()

    # Determine if LSU was home or away
    df["lsu_is_home"] = df["home_team"] == TEAM

    # LSU points and opponent points
    df["lsu_points"] = df.apply(
        lambda r: r["home_points"] if r["lsu_is_home"] else r["away_points"],
        axis=1,
    )
    df["opp_points"] = df.apply(
        lambda r: r["away_points"] if r["lsu_is_home"] else r["home_points"],
        axis=1,
    )
    df["opponent"] = df.apply(
        lambda r: r["away_team"] if r["lsu_is_home"] else r["home_team"],
        axis=1,
    )

    # Margin and result
    df["margin"] = df["lsu_points"] - df["opp_points"]
    df["win"] = (df["margin"] > 0).astype(int)
    df["total_points"] = df["lsu_points"] + df["opp_points"]

    # Season and week for grouping
    df["season"] = df["season"].astype(int)
    df["week"] = df["week"].astype(int)

    return df

In [ ]:
games_df = engineer_features(games_df)

print("Sample of engineered data:")
games_df[["season", "week", "opponent", "lsu_points", "opp_points", "margin", "win"]].head(10)

## 3. Season Summary Table

Aggregates wins, losses, win percentage, average points scored/allowed, and average margin by season.

In [ ]:
season_summary = (
    games_df.groupby("season")
    .agg(
        games=("win", "count"),
        wins=("win", "sum"),
        avg_lsu_pts=("lsu_points", "mean"),
        avg_opp_pts=("opp_points", "mean"),
        avg_margin=("margin", "mean"),
    )
    .reset_index()
)
season_summary["losses"] = season_summary["games"] - season_summary["wins"]
season_summary["win_pct"] = (season_summary["wins"] / season_summary["games"]).round(3)

season_summary

## 4. Visualizations

Four-panel chart in LSU purple and gold showing:
- Win percentage by season
- Scoring trends over time
- Distribution of game margins
- Home vs. away win rates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LSU Tigers Football: 2015-2025", fontsize=16, fontweight="bold")

# Plot 1: Win percentage by season
ax1 = axes[0, 0]
colors = ["#461D7C" if wp >= 0.5 else "#FDD023" for wp in season_summary["win_pct"]]
ax1.bar(season_summary["season"], season_summary["win_pct"], color=colors, edgecolor="white")
ax1.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
ax1.set_title("Win Percentage by Season")
ax1.set_ylabel("Win %")
ax1.set_ylim(0, 1)

# Plot 2: Points scored vs allowed by season
ax2 = axes[0, 1]
ax2.plot(
    season_summary["season"],
    season_summary["avg_lsu_pts"],
    "o-",
    color="#461D7C",
    label="LSU Avg Pts",
    linewidth=2,
)
ax2.plot(
    season_summary["season"],
    season_summary["avg_opp_pts"],
    "s--",
    color="#FDD023",
    label="Opp Avg Pts",
    linewidth=2,
)
ax2.set_title("Scoring Trends")
ax2.set_ylabel("Points per Game")
ax2.legend()

# Plot 3: Distribution of margins
ax3 = axes[1, 0]
ax3.hist(
    games_df["margin"],
    bins=25,
    color="#461D7C",
    edgecolor="white",
    alpha=0.8,
)
ax3.axvline(x=0, color="red", linestyle="--", linewidth=2, label="Break-even")
ax3.set_title("Distribution of Game Margins")
ax3.set_xlabel("Point Margin (+ = LSU win)")
ax3.set_ylabel("Frequency")
ax3.legend()

# Plot 4: Home vs Away performance
ax4 = axes[1, 1]
home_away = (
    games_df.groupby("lsu_is_home")
    .agg(win_pct=("win", "mean"), avg_margin=("margin", "mean"))
    .reset_index()
)
home_away["label"] = home_away["lsu_is_home"].map({True: "Home", False: "Away"})
ax4.bar(home_away["label"], home_away["win_pct"], color=["#FDD023", "#461D7C"], edgecolor="white")
ax4.set_title("Win % by Venue")
ax4.set_ylabel("Win %")
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.savefig("lsu_football_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to lsu_football_eda.png")

## 5. Pull Detailed Team Stats (Optional)

Per-game team stats for deeper analysis. Uncomment the call to execute.

In [ ]:
def get_lsu_team_stats(years=YEARS):
    """Pull per-game team stats for LSU."""
    all_stats = []

    with cfbd.ApiClient(configuration) as api_client:
        stats_api = cfbd.GamesApi(api_client)

        for year in years:
            try:
                stats = stats_api.get_game_team_stats(year=year, team=TEAM)
                for game in stats:
                    game_dict = game.to_dict()
                    all_stats.append(game_dict)
            except Exception as e:
                print(f"  {year} stats error: {e}")

    return all_stats

# Uncomment below to pull detailed stats (uses more API calls)
# team_stats = get_lsu_team_stats()
# print(f"Pulled stats for {len(team_stats)} games")

## 6. Pull Recruiting Data (Optional)

LSU recruiting class ratings. Uncomment the call to execute.

In [ ]:
def get_lsu_recruiting(years=YEARS):
    """Pull LSU recruiting class ratings."""
    all_recruiting = []

    with cfbd.ApiClient(configuration) as api_client:
        recruiting_api = cfbd.RecruitingApi(api_client)

        for year in years:
            try:
                classes = recruiting_api.get_recruiting_teams(year=year, team=TEAM)
                for c in classes:
                    all_recruiting.append(c.to_dict())
            except Exception as e:
                print(f"  {year} recruiting error: {e}")

    return pd.DataFrame(all_recruiting)

# Uncomment below to pull recruiting data
# recruiting_df = get_lsu_recruiting()
# recruiting_df[["year", "rank", "points"]]

## 7. Simple Predictive Model

A naive GradientBoosting classifier predicting win/loss from season, week, and home/away. Intentionally basic — the real value comes from enriching with recruiting rankings, opponent strength, betting lines, etc.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Build a simple win/loss classifier from available game-level features
model_df = games_df[["season", "week", "lsu_is_home", "win"]].dropna()
model_df["lsu_is_home"] = model_df["lsu_is_home"].astype(int)

X = model_df[["season", "week", "lsu_is_home"]]
y = model_df["win"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Naive model accuracy: {accuracy:.1%}")
print("(This baseline uses only season, week, and home/away.)")
print(f"\nFeature importances:")
for feat, imp in zip(X.columns, clf.feature_importances_):
    print(f"  {feat:15s}: {imp:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

## Project Ideas for Deeper Analysis

1. **Win Probability Model** — Pull play-by-play data, build a live win probability model. Features: score differential, time remaining, field position, down/distance.

2. **Recruiting vs. Performance** — Correlate recruiting class rank/points with wins 2-3 years later. Which position groups matter most?

3. **Opponent-Adjusted Metrics** — Use SP+ or SRS ratings to adjust raw stats. How does LSU perform vs. top-25 opponents vs. the rest?

4. **Betting Market Analysis** — Pull historical betting lines from the API. Has LSU been a profitable ATS bet? Under/over trends?

5. **Coaching Impact** — Compare offensive/defensive efficiency across coaching eras (Orgeron, Kelly, etc.)